# STARE

Source to Target Automatic Rotating Estimation (STARE) is a publicly available, blood-free quantification approach for PET tracers with irreversible kinetics. See the [preprint](https://biorxiv.org/content/10.1101/2021.09.15.460504v1.full) or [publication](https://doi.org/10.1016/j.neuroimage.2022.118901) for more details. This repository is the python implementation of it. This notebook is an example of its use.


In [1]:
""" Preliminaries, for developing and debugging in a python notebook.
    None of this is strictly necessary, but is useful for development.
"""

# Set up the path so our code is accessible w/o installing from pypi.
import sys, os
if os.path.abspath("..") not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))

# Force automatic re-load of libraries on every call
# (useful for accelerated debugging of oft-changed library code)
%load_ext autoreload
%autoreload 2

print(f"Executing notebook from '{os.getcwd()}'")


Executing notebook from '/home/mike/Projects/pystarepet/examples'


In [11]:
""" Inputs
"""

import pathlib
import pandas as pd


# Base input data
in_path = pathlib.Path("/home/mike/stare_data/sampleData")
subject_id = "NHFDG002"

# Load timing numbers and label this Series as "t"
mid_times_file = in_path / subject_id / f"{subject_id}.raw.midtime.txt"
time_data = pd.read_csv(mid_times_file, header=None, index_col=None)
time_data.columns = ["t", ]

# Find images for each time point
input_data_path = in_path / subject_id / "moco"
time_data["image"] = sorted([str(img) for img in input_data_path.glob(f"{subject_id}.*.MCFI.img")])
time_data["header"] = [str(img).replace(".img", ".hdr") for img in time_data["image"]]

for row in [0, 1, "...", len(time_data)-2, len(time_data)-1]:
    if row == "...":
        print(row)
    else:
        print("@t={:0.3f}: hdr={}, img={}".format(
            time_data.loc[row, "t"],
            time_data.loc[row, 'header'].split("/")[-1],
            time_data.loc[row, 'image'].split("/")[-1],
        ))


@t=0.125: hdr=NHFDG002.01.MCFI.hdr, img=NHFDG002.01.MCFI.img
@t=0.375: hdr=NHFDG002.02.MCFI.hdr, img=NHFDG002.02.MCFI.img
...
@t=45.000: hdr=NHFDG002.25.MCFI.hdr, img=NHFDG002.25.MCFI.img
@t=55.000: hdr=NHFDG002.26.MCFI.hdr, img=NHFDG002.26.MCFI.img


In [12]:
time_data.sample(4)

,t,image,header
4,1.125,/home/mike/stare_data/sampleData/NHFDG002/moco...,/home/mike/stare_data/sampleData/NHFDG002/moco...
18,9.500,/home/mike/stare_data/sampleData/NHFDG002/moco...,/home/mike/stare_data/sampleData/NHFDG002/moco...
20,17.500,/home/mike/stare_data/sampleData/NHFDG002/moco...,/home/mike/stare_data/sampleData/NHFDG002/moco...
1,0.375,/home/mike/stare_data/sampleData/NHFDG002/moco...,/home/mike/stare_data/sampleData/NHFDG002/moco...


In [14]:
# Load tacs numbers
tacs_file = in_path / subject_id / f"{subject_id}.TACs"
df_tacs = pd.read_csv(tacs_file, header=0, index_col=None, sep='\t')
df_tacs

,MidTime,cerfullc_c,cinr_r,pfcl_l,pfcr_r,pipl_l,pipr_r,cinl_l,parl_l,parr_r,...,gcircs_c,cerfullcs_c,hipl_l,hipr_r,cin,hip,par,pcn,pfc,pip
0,0.125,0.001219,0.000025,0.000437,0.001262,0.002146,0.002407,0.000958,0.000952,0.000821,...,0.000096,0.001082,0.000994,0.001990,0.000491,0.001492,0.000887,0.000509,0.000850,0.002276
1,0.375,0.000589,0.003880,0.001482,0.002299,0.001349,0.000853,0.000851,0.001545,0.000893,...,0.000787,0.001167,0.004580,0.002373,0.002366,0.003477,0.001219,0.001535,0.001891,0.001101
2,0.625,0.022178,0.021458,0.019049,0.019636,0.020974,0.024984,0.031402,0.021735,0.021096,...,0.018048,0.021654,0.019885,0.021534,0.026430,0.020710,0.021415,0.020786,0.019342,0.022979
3,0.875,0.065896,0.054349,0.059903,0.062334,0.051660,0.068476,0.069804,0.059352,0.057218,...,0.048731,0.066733,0.072081,0.064423,0.062076,0.068252,0.058285,0.065872,0.061118,0.060068
4,1.125,0.105287,0.077814,0.087844,0.087331,0.081134,0.087697,0.101685,0.091817,0.089255,...,0.093518,0.105841,0.101243,0.095932,0.089750,0.098588,0.090536,0.095227,0.087587,0.084416
5,1.375,0.125409,0.083804,0.106693,0.099249,0.105902,0.103579,0.115707,0.102087,0.102574,...,0.097591,0.120140,0.085125,0.088871,0.099756,0.086998,0.102330,0.107754,0.102971,0.104741
6,1.625,0.129880,0.106786,0.111563,0.108825,0.101996,0.101189,0.092756,0.114690,0.113449,...,0.127700,0.127790,0.117475,0.092421,0.099771,0.104948,0.114070,0.116796,0.110194,0.101592
7,1.875,0.133668,0.105820,0.126444,0.109873,0.122822,0.098220,0.108074,0.120471,0.124852,...,0.122860,0.132344,0.100262,0.101986,0.106947,0.101124,0.122662,0.143024,0.118159,0.110521
8,2.250,0.136150,0.119279,0.132019,0.133018,0.116210,0.105676,0.123670,0.132873,0.131268,...,0.138279,0.131453,0.117198,0.121995,0.121475,0.119596,0.132071,0.148836,0.132519,0.110943
9,2.750,0.160165,0.119490,0.144573,0.143370,0.125595,0.124995,0.140819,0.149042,0.145464,...,0.162653,0.154188,0.125031,0.117951,0.130154,0.121491,0.147253,0.157012,0.143972,0.125295


## Equations

### 1

The standard 2 tissue irreversible compartment model (2TCirr) (manuscript eq 1)

$$C_T(t)=K_1({IRF}\otimes{C_p})=K_1[(\frac{k_2}{k_2+k_3}e^{-(k_2+k_3)t} + \frac{k_3}{k_2+k_3})\otimes{C_p}](t)$$

where $t$ is time

where ${IRF}$ is the impulse response function for the target region

where $k_1$, $k_2$, $k_3$ are the microparameters of the 2TCirr model for the target region


### 2

Using equation 1 for both "source" and "target" compartments.

$$C_T(t)=K_{1,T}({IRF}_T\otimes{C_p})=K_{1,T}[(\frac{k_{2,T}}{k_{2,T}+k_{3,T}}e^{-(k_{2,T}+k_{3,T})t} + \frac{k_{3,T}}{k_{2,T}+k_{3,T}})\otimes{C_p}](t)$$

$$C_S(t)=K_{1,S}({IRF}_S\otimes{C_p})=K_{1,S}[(\frac{k_{2,S}}{k_{2,S}+k_{3,S}}e^{-(k_{2,S}+k_{3,S})t} + \frac{k_{3,S}}{k_{2,S}+k_{3,S}})\otimes{C_p}](t)$$


### 3

Define the source-to-target tissue model using a LaPlace transform to express each target compartment's TAC

$$f_T(t,\theta_{T,S})=\frac{K_{1,T}}{K_{1,S}}C_s(t) + \frac{K_{1,T}}{K_{1,S}}C_s(t)\otimes(L_{T,S}e^{{v_{T,S}}^t}+M_{T,S}e^{{\epsilon_{T,S}}^t})$$

where t is time, $\otimes$ denotes convolution, and $\theta_{T,S}$ comprises the macro-parameters $L_{T,S}$, $M_{T,S}$, $v_{T,S}$, and $\epsilon_{T,S}$.


Just checking: $\sigma\epsilon\omega\pi$ $\Sigma\Omega\Pi$

### 4

Sum the square of the residuals to determine goodness of fit.

$$\phi(t_m,\theta_{T,S})=\sum_{T=1}^{n}\sum_{m=1}^{n}w_m(C_T(t_m)-f_T(t_m,\theta_{T,S}))^2$$

where $w$ represents a set of known weights for PET frame durations.


### 5

To equation 4, add a term anchoring the estimate to PET-derived measures of vascular activity.

$$\phi(t_m,\theta_{T,S})=\sum_{T=1}^{n}\sum_{m=1}^{n}w_m(C_T(t_m)-f_T(t_m,\theta_{T,S}))^2 + \lambda\sum_{T=1}^N\lvert{K_{i,T}-K_{i,vasc,T}}\rvert$$

This is the final cost function used to fit the model.


### Altogether

From the manuscript, Figure 1

![Manuscript Figure 1](bartlett_2021_stare_figure1.jpg "Figure 1")


In [2]:
from src.util import f

f("Working")


Working
